# 3d-point-cloud (slotformer branch) -- Colab training

Backbone-architecture comparison series: the original 4 dense/sparse/SlotFormer-depth
experiments (`configs/exp_*.yaml`) plus 3 newer SlotFormer-backbone series under
`experiments/` (see each folder's own config header comments for the architecture):

- `exp1_simple_slotformer` -- original single-stage downsample backbone + external
  SlotFormer, varying SlotFormer depth (3L / 6L)
- `exp2_down_slot_up` -- 4-stage downsample encoder -> SlotFormer at the bottleneck
  -> M-stage upsample decoder, varying SlotFormer depth (3L/6L) x upsample depth
  (2/3/4)
- `exp3_unet_slotformer` -- full encoder-decoder, SlotFormer(3L) replacing residual
  blocks in every stage, varying depth (2/3/4 layers)

`run_all_experiments.py` is the single command surface for all of these -- pick one
by name (`--only <name>`), a whole series (`--group <group>`), or run everything.
Every run gets its own `checkpoints/<name>/` (large, .gitignore'd) and
`logs/<name>/loss_history.csv` (small, per-step + per-epoch train/val loss --
this is what you evaluate performance from afterward, see step 8).

**Before running anything for real**: every config's `BATCH_SIZE` is only measured
on an RTX 2070 (Windows) -- Colab GPUs (T4/A100/etc) are different hardware, so
run the batch-size/speed check (step 5) on whichever experiment you're about to
train, first.

Runtime -> Change runtime type -> GPU, before running anything below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Clone and install

The model code lives on the `slotformer` branch specifically.

In [ ]:
import os

# Safe to re-run this cell any number of times, from any state (fresh /content,
# already inside the repo, or already cloned but not cd'ed in) -- avoids the
# nested-clone trap (3d-point-cloud/3d-point-cloud/...) that a plain
# `!git clone && %cd` gives you if you re-run this cell after the first `%cd`
# already moved you inside the repo.
if os.path.basename(os.getcwd()) == "3d-point-cloud" and os.path.exists("train.py"):
    print("already inside 3d-point-cloud/ -- nothing to do")
else:
    if not os.path.isdir("3d-point-cloud"):
        !git clone -b slotformer https://github.com/izione/3d-point-cloud.git
    else:
        print("3d-point-cloud/ already exists here -- skipping clone")
    %cd 3d-point-cloud

!pip install -q -r requirements.txt

Optional: spconv accelerates the sparse backbones if it installs cleanly for this
Colab image's CUDA version -- everything works without it too (pure-PyTorch
fallback, `models/backbone3d_auto.py` probes automatically). Note the four newer
SlotFormer backbones (`sparse_down_slot_up`, `sparse_slot_stages`, `sparse_slot_unet`,
`sparse_slot_light_unet`) always use the pure-PyTorch path regardless -- spconv only
helps the original `auto`-typed backbone (experiments 1-4).

In [ ]:
# !pip install -q spconv-cu126   # pick the cuXXX tag matching this runtime's CUDA (see https://github.com/traveller59/spconv)

## 2. Get the dataset

Downloads `dataset.zip` from a Google Drive share link and unzips it onto the Colab
VM's local disk (`/content/dataset_extracted`) -- not the Drive-mounted path,
since `SonarDiverDataset` reads many small files per epoch and local disk is much
faster than Drive's network filesystem for that access pattern.

In [ ]:
DATASET_ZIP_SHARE_URL = "https://drive.google.com/file/d/1JTnVQ1c25z2MfJQUvIHz4dgyRsFiPLm_/view?usp=drive_link"

In [ ]:
!pip install -q gdown
!gdown --fuzzy "{DATASET_ZIP_SHARE_URL}" -O /content/dataset.zip
!unzip -q -o /content/dataset.zip -d /content/dataset_extracted
!ls /content/dataset_extracted

Auto-detect `DATASET_ROOT` (descends past wrapper folders until it finds
`Person*` folders) and patch `configs/default.yaml`'s `DATA.ROOT` to point at it.

In [ ]:
import os
import yaml

DATASET_ROOT = "/content/dataset_extracted"
while True:
    entries = [e for e in os.listdir(DATASET_ROOT) if not e.startswith(".")]
    if any(e.startswith("Person") for e in entries):
        break
    subdirs = [e for e in entries if os.path.isdir(os.path.join(DATASET_ROOT, e))]
    if len(subdirs) != 1:
        raise RuntimeError(
            f"couldn't auto-detect DATASET_ROOT under {DATASET_ROOT!r} -- "
            f"found {entries!r}, expected exactly one wrapper folder or Person* folders directly. "
            f"Set DATASET_ROOT by hand and skip this cell's loop."
        )
    DATASET_ROOT = os.path.join(DATASET_ROOT, subdirs[0])
print(f"DATASET_ROOT = {DATASET_ROOT}")

with open("configs/default.yaml") as f:
    cfg_text = f.read()
cfg = yaml.safe_load(cfg_text)
cfg["DATA"]["ROOT"] = DATASET_ROOT
with open("configs/default.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("patched configs/default.yaml DATA.ROOT ->", DATASET_ROOT)

No shareable zip link yet, or prefer the manual route? Mount your own Drive copy
instead (slower per-epoch, no zip/link needed):

```python
from google.colab import drive
drive.mount('/content/drive')
DATASET_ROOT = "/content/drive/MyDrive/dataset"  # wherever you uploaded PersonX/scene_XXXX/
# ... then run the DATA.ROOT-patching cell above with this DATASET_ROOT
```

## 3. Mount Drive (for checkpoint/log persistence)

Separate from the dataset above -- this is so checkpoints AND the loss-history CSVs
survive a Colab disconnect (step 6 writes both there).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Sanity check (synthetic data, no dataset needed)

Confirms every `configs/exp_*.yaml` AND every `experiments/*/*.yaml` constructs,
runs forward/backward, and produces finite losses, before touching the real dataset.

In [ ]:
!python smoke_test.py

## 5. Pick an experiment + quick batch-size/speed check

Set `EXPERIMENT_NAME` to one of `run_all_experiments.py`'s names (printed below) --
this is the single place that controls which experiment the rest of the notebook
trains/evaluates. Re-run this cell after changing `EXPERIMENT_NAME` to re-check
memory/speed for the new choice before committing to a long run.

In [ ]:
!python run_all_experiments.py --dry_run 2>&1 | grep -oP '(?<=\[)[^\]]+(?=\])' | sort -u

In [ ]:
EXPERIMENT_NAME = "exp2_4down_3up_6l"   # <-- change this to any name from the list printed above

import subprocess
out = subprocess.run(["python", "run_all_experiments.py", "--dry_run"], capture_output=True, text=True).stdout
CONFIG_PATH = None
for line in out.splitlines():
    if f"/{EXPERIMENT_NAME}]" in line:
        CONFIG_PATH = line.split("--config", 1)[1].split("--ckpt_dir", 1)[0].strip()
        break
if CONFIG_PATH is None:
    raise ValueError(f"{EXPERIMENT_NAME!r} not found in run_all_experiments.EXPERIMENTS -- check the name list above")
print(f"EXPERIMENT_NAME={EXPERIMENT_NAME}  CONFIG_PATH={CONFIG_PATH}")

In [ ]:
import time

import torch
from torch.utils.data import DataLoader

from config_utils import load_config
from data.dataset import SonarDiverDataset, collate_fn
from models.detector import DiverDetector

BATCH_SIZE_TO_TEST = None   # None = read OPTIMIZATION.BATCH_SIZE from the config; set an int here to override
N_WARMUP = 5    # excluded from timing (first-call overhead: cuDNN autotune, allocator growth, etc.)
N_TIMED = 30

device = torch.device("cuda")
cfg = load_config(CONFIG_PATH)
if BATCH_SIZE_TO_TEST is None:
    BATCH_SIZE_TO_TEST = cfg["OPTIMIZATION"]["BATCH_SIZE"]
ds = SonarDiverDataset(cfg, "train")
loader = DataLoader(ds, batch_size=BATCH_SIZE_TO_TEST, shuffle=True, collate_fn=collate_fn, drop_last=True)
model = DiverDetector(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
steps_per_epoch = len(ds) // BATCH_SIZE_TO_TEST
print(f"train frames: {len(ds)}  steps/epoch at batch={BATCH_SIZE_TO_TEST}: {steps_per_epoch}")

def run_one_step(it):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader)
        batch = next(it)
    losses, pred, stem_coords, assign_result = model.loss(batch, device)
    optimizer.zero_grad()
    losses["total"].backward()
    optimizer.step()
    return it, stem_coords.shape[0]

it = iter(loader)
torch.cuda.reset_peak_memory_stats()
for _ in range(N_WARMUP):
    it, _ = run_one_step(it)

torch.cuda.synchronize()
t0 = time.perf_counter()
last_n_voxels = None
for _ in range(N_TIMED):
    it, last_n_voxels = run_one_step(it)
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

sec_per_step = elapsed / N_TIMED
reserved = torch.cuda.max_memory_reserved() / 1024**3
total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
epoch_min = sec_per_step * steps_per_epoch / 60
num_epochs = cfg["OPTIMIZATION"]["NUM_EPOCHS"]

print(f"peak_reserved={reserved:.2f} GiB / {total_mem:.1f} GiB ({100*reserved/total_mem:.0f}%)  "
      f"stem_voxels(last step)={last_n_voxels}")
print(f"{sec_per_step*1000:.0f} ms/step  ->  ~{epoch_min:.1f} min/epoch  ->  "
      f"~{epoch_min*num_epochs/60:.1f} hours for all {num_epochs} epochs (NUM_EPOCHS in the config)")

del model, optimizer, loader, ds
torch.cuda.empty_cache()

## 6. Train

Checkpoints AND the loss-history CSV go to Drive so a disconnect mid-run doesn't
lose progress. This mirrors `run_all_experiments.py`'s own
`checkpoints/<name>/` + `logs/<name>/loss_history.csv` layout, just rooted on Drive
instead of the ephemeral Colab disk.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/3d-point-cloud-runs"

!python train.py --config "{CONFIG_PATH}" \
    --ckpt_dir "{DRIVE_ROOT}/checkpoints/{EXPERIMENT_NAME}" \
    --log_file "{DRIVE_ROOT}/logs/{EXPERIMENT_NAME}/loss_history.csv" \
    --exp_name "{EXPERIMENT_NAME}"

Resume after a disconnect (picks up the schedule/step count from the checkpoint):

In [ ]:
# !python train.py --config "{CONFIG_PATH}" \
#     --ckpt_dir "{DRIVE_ROOT}/checkpoints/{EXPERIMENT_NAME}" \
#     --log_file "{DRIVE_ROOT}/logs/{EXPERIMENT_NAME}/loss_history.csv" \
#     --exp_name "{EXPERIMENT_NAME}" \
#     --resume "{DRIVE_ROOT}/checkpoints/{EXPERIMENT_NAME}/{EXPERIMENT_NAME}_last.pth"

## 7. Optional: run a whole series (or everything) back-to-back

`run_all_experiments.py --group <name>` runs every experiment in one series
(`exp1_simple_slotformer` / `exp2_down_slot_up` / `exp3_unet_slotformer` /
`original4`); omit `--group`/`--only` to run literally everything. Point Drive
paths at each run the same way as step 6 by editing `run_all_experiments.py`'s
`ckpt_dir`/`log_file` lines, or just let it write to the local (ephemeral)
`checkpoints/`/`logs/` and copy them to Drive yourself afterward if you don't
need per-run resume safety.

In [ ]:
# !python run_all_experiments.py --only exp2_4down_3up_6l
# !python run_all_experiments.py --group exp2_down_slot_up   # all 6 in that series
# !python run_all_experiments.py                              # literally everything (15 experiments)

## 8. Evaluate

`logs/<name>/loss_history.csv` (or the Drive path from step 6) has per-step and
per-epoch train/val loss already -- open it directly (pandas/plot) for a first
look. For AP@IoU(0.30/0.35/0.40) on the held-out test split:

```bash
!python test.py --checkpoint "{DRIVE_ROOT}/checkpoints/{EXPERIMENT_NAME}/{EXPERIMENT_NAME}_last.pth" --split test
```

See `README.md`'s "Test / evaluate" section for `--pr_curve_out` (PR-curve-per-IoU
plot) and `eval_pr_comparison.py` for comparing multiple checkpoints (e.g. two
experiments from this series) on one figure -- each checkpoint carries its own
`cfg` (saved by `train.py`), so comparisons stay correct even if `configs/default.yaml`
changes later.